# Gemma 4 E2B — Developer Tutorial

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/drive/17p8azP2UZxotkd7zkgp3yChNriqVphcc?usp=sharing)

> **Gemma 4 E2B** is Google's latest open multimodal model with **thinking mode**, capable of reasoning over **text, images, and audio** — all in one model.

## What you'll learn
In this notebook, you will learn how to:
1. Authenticate with HuggingFace to download gated models
2. Load `google/gemma-4-E2B-it` with `transformers`
3. Run text generation **with** and **without** thinking mode
4. Run **image understanding** (visual question answering)
5. Run **audio transcription and translation**

## Prerequisites
Before running this notebook:
- **GPU Runtime**: Go to `Runtime → Change runtime type → T4 GPU`
- **HuggingFace Token**: Add your HF token to Colab Secrets with key name `HF_TOKEN`
  - Get your token at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)
  - Accept the Gemma 4 model license at [huggingface.co/google/gemma-4-E2B-it](https://huggingface.co/google/gemma-4-E2B-it)

---


In [ ]:
MODEL_STRING = "google/gemma-4-E2B-it"
print(f"Model: {MODEL_STRING}")

## Step 2 — Authenticate with HuggingFace

Gemma 4 is a **gated model**, which means you need to:
1. Accept the license on HuggingFace
2. Provide your HF token to authenticate

We use **Colab Secrets** (the key icon in the left sidebar) to securely store the token — never hardcode tokens in your notebook.


In [ ]:
from google.colab import userdata
from huggingface_hub import login

# Reads HF_TOKEN from Colab Secrets (left sidebar → key icon)
hf_token = userdata.get('HF_TOKEN')
login(hf_token)
print("✅ Authenticated with HuggingFace.")

## Step 3 — Install Transformers

Gemma 4 requires **transformers >= 5.5.0** for the new multimodal processor and thinking mode support.

> **Note:** After installation, the kernel may ask you to restart. If so, restart and run all cells again from Step 1.


In [ ]:
!pip -q install transformers==5.5.0

import transformers
print(f"✅ Transformers version: {transformers.__version__}")

## Step 4 — Load the Model & Processor

We load two components:
- **`AutoProcessor`** — handles tokenization + multimodal inputs (text, image, audio)
- **`AutoModelForCausalLM`** — the Gemma 4 language model weights

We use `dtype=torch.bfloat16` to reduce memory usage (~half of float32) and `device_map="auto"` to automatically place layers on available GPU(s).

> **Estimated download size:** ~10 GB. This will take a few minutes on first run.


In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForCausalLM

MODEL_ID = MODEL_STRING

print(f"Loading processor for {MODEL_ID}...")
processor = AutoProcessor.from_pretrained(MODEL_ID)

print("Loading model weights (this may take a few minutes)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,   # use bfloat16 to halve GPU memory usage
    device_map="auto",       # auto-place on GPU(s)
)

print(f"✅ Model loaded on: {model.device}")
print(f"   Parameters: {model.num_parameters() / 1e9:.1f}B")

## 4. Generate (with Thinking)

In [ ]:
from IPython.display import display, Markdown

# ✏️ Change this prompt to anything you like
prompt = "What are some common use cases for Deep Learning in Finance?" #@param {type:"string"}

# Format as a chat message
messages = [{"role": "user", "content": prompt}]

# Apply chat template with thinking mode ENABLED
text = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True,   # <-- enables step-by-step reasoning
)

# Tokenize and move to GPU
inputs = processor(text=text, return_tensors='pt').to(model.device)
input_len = inputs["input_ids"].shape[-1]

# Generate response
with torch.inference_mode():
    outputs = model.generate(**inputs, max_new_tokens=4000)

# Decode only the new tokens (skip the input prompt)
raw_response = processor.decode(outputs[0][input_len:], skip_special_tokens=False)

# Parse into thinking + answer
result = processor.parse_response(raw_response)

print("=== 💭 Model's Thinking Process ===\n")
display(Markdown(result['thinking']))

print("\n=== ✅ Final Answer ===\n")
display(Markdown(result['content']))

In [11]:
from IPython.display import display, Markdown

result = processor.parse_response(response)

if result["thinking"]:
    print("\n=== Thoughts ===\n")
    display(Markdown(result['thinking']))
    print("\n=== Answer ===\n")
    display(Markdown(result['content']))
else:
    print(f"\n=== Response (no thinking delimiters) ===\n")
    display(Markdown(response))


=== Thoughts ===



Here's a thinking process that leads to the suggested answer:

1.  **Deconstruct the Request:** The user is asking for "common use cases for Deep Learning (DL) in Finance." This requires identifying areas within finance where complex data, non-linear relationships, or high-volume time series analysis are crucial.

2.  **Identify Core Areas of Finance:** What does finance *do*?
    *   Trading/Markets (Prediction, execution)
    *   Risk Management (Modeling volatility, credit)
    *   Customer Behavior (Lending decisions, churn)
    *   Asset Management (Portfolio optimization)
    *   Fraud Detection/Compliance

3.  **Map Deep Learning Capabilities to Finance Problems:** Deep Learning excels at:
    *   Handling unstructured data (text, news).
    *   Capturing complex, non-linear patterns in high-dimensional data (time series, market correlations).
    *   Sequence modeling (predicting future states).
    *   Generating complex representations (embeddings).

4.  **Generate Specific Use Cases (Drafting & Refinement):**

    *   *Area: Trading/Prediction*
        *   *Initial thought:* Predict stock prices. (Too simple, standard ML can do this.)
        *   *DL refinement:* Better prediction using high-frequency data, incorporating news sentiment, and handling complex market dynamics. (Use Case 1: Time Series Forecasting/Price Prediction).
        *   *DL refinement:* High-frequency trading strategies, anomaly detection in order flow.

    *   *Area: Risk Management*
        *   *Initial thought:* Credit scoring. (Standard ML/Logistic Regression is common.)
        *   *DL refinement:* Better handling of complex borrower data (unstructured documents, transaction history) and modeling systemic risk. (Use Case 2: Credit Risk Modeling).
        *   *DL refinement:* Stress testing and scenario generation.

    *   *Area: Customer/Lending*
        *   *Initial thought:* Loan approval.
        *   *DL refinement:* Using NLP on applications, handling behavioral data for dynamic pricing. (Use Case 3: Fraud Detection/AML).
        *   *DL refinement:* Personalized product recommendations (robo-advisors).

    *   *Area: Alternative Data & NLP*
        *   *Initial thought:* Read news.
        *   *DL refinement:* Extracting sentiment and topic modeling from news, social media, and earnings calls to inform trading decisions. (Use Case 4: Sentiment Analysis/News Processing).

    *   *Area: Portfolio Management*
        *   *Initial thought:* Optimize portfolio.
        *   *DL refinement:* Dynamic asset allocation, optimizing risk-adjusted returns over long horizons. (Use Case 5: Portfolio Optimization).

5.  **Structure and Elaborate the Answer:** Group the use cases logically (Market, Risk, Customer, etc.) and ensure the explanation highlights *why* Deep Learning is better than traditional methods (i.e., handling complexity, unstructured data).

6.  **Add Context/Challenges (Crucial for a complete answer):** Acknowledge the practical realities (data requirements, interpretability, model training). (Self-Correction: Don't just list uses; mention the underlying technology/data type.)

7.  **Final Review:** Does the answer directly address "Deep Learning in Finance"? Yes. Is it comprehensive? Yes. (The resulting structure is clear and professional.) (This matches the final output provided below.)


=== Answer ===



Deep Learning (DL) has become increasingly powerful in finance because it excels at handling **high-dimensional, non-linear data**, recognizing complex patterns in **time series data**, and processing **unstructured data** (like text and images).

Here is a breakdown of common use cases for Deep Learning in the financial industry, categorized by functional area.

---

## 1. Algorithmic Trading and Market Prediction

This is perhaps the most visible application, leveraging DL's ability to find subtle patterns in massive datasets.

*   **Time Series Forecasting:** Using Recurrent Neural Networks (RNNs), Long Short-Term Memory (LSTM) networks, or Transformers to predict future stock prices, volatility, or commodity prices based on historical price data, trading volumes, and macroeconomic indicators.
    *   ***Why DL is used:*** Traditional models often struggle to capture long-term dependencies and complex, non-linear relationships inherent in financial time series. LSTMs are excellent at remembering long-term patterns.
*   **Market Sentiment Analysis (News/Social Media):** Employing Natural Language Processing (NLP) models (like BERT or other transformer models) to process vast amounts of unstructured text (news articles, earnings call transcripts, social media posts, analyst reports) to derive real-time market sentiment.
    *   ***Application:*** Generating trading signals based on sudden shifts in collective market mood.
*   **High-Frequency Trading (HFT) Strategy:** Using DL to analyze raw order book data (bid/ask prices, trade volumes) in milliseconds to identify fleeting arbitrage opportunities or predict short-term price movements.
*   **Anomaly Detection in Order Flow:** Identifying unusual trading patterns or "spoofing" activity in the order book that might indicate market manipulation.

## 2. Risk Management

DL models help financial institutions move beyond simple statistical models to create more dynamic and accurate risk assessments.

*   **Credit Risk Modeling:** Using DL to analyze complex borrower data—including structured financial statements, unstructured legal documents, transaction histories, and even geographical data—to determine the probability of default (PD) with greater accuracy than traditional logistic regression models.
*   **Market Risk Modeling (Stress Testing):** Building complex neural networks to simulate systemic risk and model how different market factors (interest rates, geopolitical events, currency fluctuations) interact to affect the firm’s portfolio.
*   **Liquidity Risk Prediction:** Forecasting potential cash flow shortfalls or liquidity crises by analyzing complex interdependencies between various asset classes and market conditions.

## 3. Fraud Detection and Compliance (AML)

Deep Learning is highly effective at identifying subtle deviations from normal behavior in high-volume transaction streams.

*   **Transaction Fraud Detection:** Using sequence models and autoencoders to analyze user behavior (login times, transaction amounts, geographical locations) in real-time. DL can detect sophisticated fraud rings that involve complex, multi-stage transactions.
*   **Anti-Money Laundering (AML):** Identifying suspicious patterns in large sets of transactions that might indicate money laundering activities, especially by linking seemingly unrelated accounts or entities.
*   **Regulatory Compliance:** Automating the review of vast quantities of regulatory documents and communication to ensure adherence to complex compliance standards (e.g., Basel III requirements).

## 4. Customer Experience and Portfolio Management

DL improves personalization and optimization for clients.

*   **Personalized Robo-Advising:** Using collaborative filtering and deep neural networks to analyze a client’s risk tolerance, financial goals, and behavioral patterns to construct highly personalized, dynamically rebalanced investment portfolios.
*   **Churn Prediction:** Identifying customers who are likely to leave a bank or investment platform by analyzing their interaction patterns, service requests, and financial behavior.
*   **Dynamic Pricing:** Optimizing pricing models for financial products (like insurance or loans) by considering complex variables in real-time to maximize revenue while minimizing risk.

---

## Summary Table: DL vs. Traditional Methods in Finance

| Use Case Area | Traditional Approach | Deep Learning Approach | Key Advantage of DL |
| :--- | :--- | :--- | :--- |
| **Market Prediction** | Linear Regression, ARIMA | LSTMs, Transformers | Captures long-term, non-linear dependencies and sequence history. |
| **Credit Scoring** | Logistic Regression, Decision Trees | Deep Neural Networks (DNNs) | Effectively processes complex, unstructured data (documents, text). |
| **Fraud Detection** | Rule-based systems, Simple Clustering | Autoencoders, RNNs | Detects subtle, high-dimensional anomalies in massive data streams. |
| **Portfolio Mgmt.** | Mean-Variance Optimization | Reinforcement Learning (RL) | Optimizes complex, dynamic allocation strategies over time. |
| **Sentiment Analysis** | Lexicon-based methods | NLP (BERT, GPT) | Understands context, sarcasm, and nuance in human language. |

## Key Challenges of Using Deep Learning in Finance

While powerful, implementing DL in finance faces significant hurdles:

1.  **Data Requirements:** DL models are data-hungry. High-quality, labeled data (especially for fraud or credit risk) is often scarce or highly sensitive.
2.  **Interpretability (The Black Box Problem):** Financial regulators and risk managers often require explanations for model decisions. Deep Learning models can be "black boxes," making it difficult to explain *why* a model made a specific trading or credit decision. Techniques like SHAP values are increasingly used to mitigate this.
3.  **Model Stability:** Financial markets are non-stationary (patterns change over time). DL models must be constantly retrained and monitored to prevent decay in predictive performance.
4.  **Computational Cost:** Training complex deep learning models requires substantial computational resources (GPUs), which can be expensive for large financial institutions.

In [ ]:
from IPython.display import display, Markdown

# ✏️ Try your own prompt here — thinking mode shines on reasoning-heavy questions
prompt = "I have a list of 1 million integers. Walk me through the most efficient way to find all duplicate values and explain the time and space complexity of your approach." #@param {type:"string"}

messages = [{"role": "user", "content": prompt}]

# Apply chat template with thinking mode ENABLED
text = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True,   # <-- enables step-by-step reasoning
)

inputs = processor(text=text, return_tensors='pt').to(model.device)
input_len = inputs["input_ids"].shape[-1]

with torch.inference_mode():
    outputs = model.generate(**inputs, max_new_tokens=4000)

# Decode only the newly generated tokens
raw_response = processor.decode(outputs[0][input_len:], skip_special_tokens=False)

# Split into thinking trace + final answer
result = processor.parse_response(raw_response)

print("=== 💭 Model's Thinking Process ===\n")
display(Markdown(result['thinking']))

print("\n=== ✅ Final Answer ===\n")
display(Markdown(result['content']))

## Step 6 — Text Generation WITHOUT Thinking Mode

When you don't need step-by-step reasoning (e.g. simple Q&A, summarization, code generation), you can disable thinking mode for **faster responses**.

Just set `enable_thinking=False`.


In [ ]:
!wget -q -O /tmp/test_image.jpg "https://storage.googleapis.com/sfr-vision-language-research/BLIP/demo.jpg"

In [ ]:
# Download a sample image
!wget -q -O /tmp/test_image.jpg "https://storage.googleapis.com/sfr-vision-language-research/BLIP/demo.jpg"
print("✅ Image downloaded.")

In [ ]:
from PIL import Image
from IPython.display import display

IMAGE_PATH = "/tmp/test_image.jpg"
image = Image.open(IMAGE_PATH)

# Preview the image
print("Input image:")
display(image)

# ✏️ Change this question to ask anything about the image
question = "What is going on in this image?" #@param {type:"string"}

# NOTE: processor.image_token is a special placeholder that tells
# the model where the image is inserted in the prompt
messages = [
    {
        'role': 'user',
        'content': [
            {'type': 'text', 'text': f'{processor.image_token}\n{question}'}
        ]
    }
]

text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# Pass both text AND image to the processor
inputs = processor(text=text, images=image, return_tensors="pt").to(model.device)

with torch.inference_mode():
    outputs = model.generate(**inputs, max_new_tokens=128)

# Decode the full output (input + generated)
answer = processor.tokenizer.decode(outputs[0], skip_special_tokens=True)

print("\n=== 🖼️ Model's Answer ===\n")
# Print only the assistant's reply (after the last "model" turn)
reply = answer.split("model\n")[-1].strip()
print(reply)

In [ ]:
# ✏️ Swap this for any quick question — no-thinking is faster for simple lookups
prompt = "Give me a Python one-liner to flatten a nested list." #@param {type:"string"}

messages = [{"role": "user", "content": prompt}]

# Apply chat template with thinking mode DISABLED
text = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,  # <-- direct answer, no reasoning trace
)

inputs = processor(text=text, return_tensors='pt').to(model.device)
input_len = inputs["input_ids"].shape[-1]

with torch.inference_mode():
    outputs = model.generate(**inputs, max_new_tokens=512)

raw_response = processor.decode(outputs[0][input_len:], skip_special_tokens=False)
result = processor.parse_response(raw_response)

print("=== ✅ Answer (no thinking) ===\n")
display(Markdown(result['content']))

In [ ]:
import requests
from IPython.display import Audio, display

AUDIO_URL = "https://github.com/voxserv/audio_quality_testing_samples/raw/master/testaudio/16000/test01_20s.wav"
AUDIO_PATH = "/content/test01_20s.wav"

response = requests.get(AUDIO_URL)
response.raise_for_status()

with open(AUDIO_PATH, "wb") as f:
    f.write(response.content)

print(f"✅ Audio downloaded: {AUDIO_PATH}")

# Preview the audio
print("\nListen to the audio clip:")
display(Audio(AUDIO_PATH))

In [ ]:
# Build the message with audio + transcription instruction
messages = [
    {
        "role": "user",
        "content": [
            {"type": "audio", "audio": AUDIO_PATH},  # pass audio file path here
            {"type": "text", "text": "Transcribe this audio file"},
        ]
    }
]

input_ids = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
)
input_ids = input_ids.to(model.device, dtype=model.dtype)

outputs = model.generate(**input_ids, max_new_tokens=64)

text_output = processor.batch_decode(
    outputs,
    skip_special_tokens=False,
    clean_up_tokenization_spaces=False,
)

print("=== 🎙️ Transcription ===\n")
# Extract only the model's reply
import re
reply = re.sub(r"^.*?<\|turn>model\n", "", text_output[0], flags=re.DOTALL).strip()
print(reply)

In [ ]:
import re

model_response = re.sub(r"^.*?<\|turn>model\n", "", text[0], flags=re.DOTALL)

from PIL import Image
from IPython.display import display

IMAGE_PATH = "/tmp/test_image.jpg"
image = Image.open(IMAGE_PATH)

print("Input image:")
display(image)

# ✏️ Ask anything about the image
question = "Describe the mood of this scene and what the subject might be doing." #@param {type:"string"}

# processor.image_token tells the model where the image appears in the prompt
messages = [
    {
        'role': 'user',
        'content': [
            {'type': 'text', 'text': f'{processor.image_token}\n{question}'}
        ]
    }
]

text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# Pass both text AND image to the processor
inputs = processor(text=text, images=image, return_tensors="pt").to(model.device)

with torch.inference_mode():
    outputs = model.generate(**inputs, max_new_tokens=128)

answer = processor.tokenizer.decode(outputs[0], skip_special_tokens=True)

print("\n=== 🖼️ Model's Answer ===\n")
reply = answer.split("model\n")[-1].strip()
print(reply)

---

## What's Next?

You've now seen all the core capabilities of Gemma 4 E2B:

| Capability | Key Parameter |
|---|---|
| Text generation | `enable_thinking=True/False` |
| Image understanding | Pass `images=image` to processor |
| Audio transcription | `{"type": "audio", "audio": path}` |
| Audio translation | Same as transcription + translated instruction |

### Ideas to try
- Swap `MODEL_STRING` for another Gemma variant (e.g. `google/gemma-4-9B-it`)
- Try your own images or audio files
- Chain outputs: transcribe audio → summarize with thinking mode
- Build a Gradio UI on top of this notebook

### Resources
- [Gemma 4 on HuggingFace](https://huggingface.co/google/gemma-4-E2B-it)
- [Transformers Docs](https://huggingface.co/docs/transformers)
- [Google Gemma Blog](https://ai.google.dev/gemma)
